# 4.5 Agent-Based Indoor Search

The task is to enable the drone to find and approach specific targets.

Let the drone autonomously search for objects.

Using object detection and depth camera capabilities, we can get each target's distance and angle relative to the drone:

[('yellow duck', 15.375, -21.425346762452097), ('flower', 14.7421875, -42.059604476553346), ('coca cola', 14.703125, -12.83171588735092), ('mirror', 13.203125, 6.831911236078974)]

The key functions are as follows:

In [ ]:
from typing import List,Tuple

def ob_objects(obj_name_list:List[str])-> List[Tuple[str, float, float]]:
    """
    Perform object localization on the drone's camera image, returning a list of [(object name, distance, angle in degrees), ...]

    Args:
        obj_name_list: List of target names (must be in English)

    Returns:
        List: [(object name, distance from drone, angle from drone in degrees), ...]
    """

    # Step 1: Object detection
    prompt = ".".join(obj_name_list)
    # obj_id_list: [obj1, obj2,...], obj_locs: [[xmin, ymin, xmax, ymax],[xmin, ymin, xmax, ymax],...]
    obj_id_list, obj_locs = detect(prompt)

    # Step 2: Get depth camera data
    responses = client.simGetImages([
        # png format
        airsim.ImageRequest(0, airsim.ImageType.Scene, pixels_as_float=False, compress=True),

        # Floating point uncompressed depth image, pixels represent distance to image plane
        airsim.ImageRequest(0, airsim.ImageType.DepthPlanar, pixels_as_float=True),

        # Pixels represent distance to camera
        airsim.ImageRequest(0, airsim.ImageType.DepthPerspective, pixels_as_float=True)
      ]
    )

    img_depth_planar = np.array(responses[1].image_data_float).reshape(responses[1].height, responses[0].width)
    img_depth_perspective = np.array(responses[2].image_data_float).reshape(responses[2].height, responses[1].width)

    # Regular image
    image_data = responses[0].image_data_uint8
    img = cv2.imdecode(np.array(bytearray(image_data), dtype='uint8'), cv2.IMREAD_UNCHANGED)  # Read from binary image data
    img = cv2.cvtColor(img, cv2.COLOR_RGBA2RGB)  # 4 channels to 3


    final_obj_list = []  # Result list with target positions
    # Build target results
    index = 0
    for bbox in obj_locs:
        center_x = int((bbox[0] + bbox[2]) / 2)
        center_y = int((bbox[1] + bbox[3]) / 2)

        depth_distance = self.img_depth_planar[center_y, center_x,]  # Distance to image plane
        camera_distance = self.img_depth_perspective[center_y, center_x]  # Distance to camera

        # Calculate angle
        angel = math.acos(depth_distance / camera_distance)
        angel_degree = math.degrees(angel)

        # Determine sign: left is negative, right is positive (yaw only)
        if center_x < self.img.shape[1] / 2:
            # If target is on the left side of the image, turn left, degree is negative
            angel_degree = -1 * angel_degree

        obj_name = phrases[index]  # Get target name (may have multiple)

        obj_info = (obj_name, camera_distance, angel_degree)
        final_obj_list.append(obj_info)
        index = index + 1

    return final_obj_list

The drone navigates toward a target based on its distance and angle. Here are the movement functions:

In [ ]:
def turn(angle: float)->str:
    """
    Rotate the drone by the specified angle

    Args:
        angle: The angle to rotate (in degrees)

    Returns:
        str: Success status description
    """
    yaw_degree = get_yaw()
    yaw_degree = yaw_degree + angle
    set_yaw(yaw_degree)
    return "success"

def move(distance: float)->str:
    """
    Move forward by the specified distance in meters

    Args:
        distance: Distance to move forward, in meters

    Returns:
        str: Success status description
    """
    step_length = distance
    cur_position = get_drone_position()
    yaw_degree = cur_position[3]
    # Convert angle to radians
    yaw = math.radians(yaw_degree)
    # Move forward by distance
    x = cur_position[0] + step_length*math.cos(yaw)
    y = cur_position[1] + step_length*math.sin(yaw)
    z = cur_position[2]
    fly_to([x, y, z, 0])
    return "success"

## Let's Begin

In [ ]:
#!pip install smolagents[litellm]
from smolagents import CodeAgent, LiteLLMModel

model = LiteLLMModel(
    model_id="openai/moonshotai/Kimi-K2.6",  # This model is a bit weak for agentic behaviours though
    api_base="https://api.intelligence.io.solutions/api/v1", # replace with 127.0.0.1:11434 or remote open-ai compatible server if necessary
    api_key="ffd77d7c-f420-4b69-8557-80e7fa85c8b9", # Replace with your own API key
    flatten_messages_as_text=True, # Some models may not support multi-step conversations well
)

In [ ]:
from airsim_smol_wrapper import *

In [ ]:
# Enter the room
takeoff()
set_yaw(90)
for _ in range(5):
    forward()

In [ ]:
agent = CodeAgent(tools=[turn, move, watch], model=model)

In [ ]:
command = """
Look around, what objects are inside the room?
"""

result = agent.run(
    command
)

In [ ]:
agent = CodeAgent(tools=[turn, move, ob_objects, watch], model=model)

In [ ]:
command = """
Look ahead about 1 meter, what objects are there?
"""

result = agent.run(
    command
)

In [ ]:
reset()